# Imports and Setup

In [12]:
# 1. Imports and environment setup
import os
import random
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights, resnet18, ResNet18_Weights

import kagglehub

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"

import os
# Création d'un dossier pour ranger les modèles si vous le souhaitez
os.makedirs("saved_models", exist_ok=True)

# Utility Functions

In [13]:
# 2. Utility and visualization functions
def plot_training_history(train_loss, val_loss, train_acc, val_acc, title="Model Training Log"):
    fig, axes = plt.subplots(ncols=2, figsize=(15, 6))

    # Loss plot
    axes[0].plot(train_loss, label="Training", marker='o', ls='--')
    axes[0].plot(val_loss, label="Validation", marker='s')
    axes[0].set_title(f"{title} - Loss")
    axes[0].set_xlabel("Epochs")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy plot
    axes[1].plot(train_acc, label="Training", marker='o', ls='--')
    axes[1].plot(val_acc, label="Validation", marker='s')
    axes[1].set_title(f"{title} - Accuracy")
    axes[1].set_xlabel("Epochs")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def distillation_loss(student_logits, labels, teacher_logits, T=3.0, alpha=0.5):
    """
    T: Temperature to smooth probabilities
    alpha: Weight given to the teacher's loss (0.5 = balanced)
    """
    # Soft loss (Knowledge Distillation)
    # Apply temperature to log-probabilities
    soft_loss = nn.KLDivLoss(reduction='batchmean')(
        nn.functional.log_softmax(student_logits / T, dim=1),
        nn.functional.softmax(teacher_logits / T, dim=1)
    ) * (T * T)

    # Hard loss (Standard classification)
    hard_loss = nn.functional.cross_entropy(student_logits, labels)

    return alpha * soft_loss + (1.0 - alpha) * hard_loss

# Dataset Preparation

In [14]:
# 3. Download and prepare the dataset
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Path to dataset files:", path)

def create_df(dataset_path):
    data_dict = {"images": [], "labels": []}
    for label in os.listdir(dataset_path):
        img_dir = os.path.join(dataset_path, label)
        if not os.path.isdir(img_dir): continue

        images = os.listdir(img_dir)
        index = 0
        for img in images:
            data_dict["images"].append(os.path.join(img_dir, img))
            data_dict["labels"].append(label)
            index += 1
            if index == 1000: # Limit to 1000 per class
                break
    return pd.DataFrame(data_dict)

train_path = "/kaggle/input/brain-tumor-mri-dataset/Training"
df = create_df(train_path)

# Label encoding
index_label = {i: label for i, label in enumerate(df["labels"].unique())}
label_index = {label: i for i, label in enumerate(df["labels"].unique())}
df["labels"] = df["labels"].map(label_index)

# Dataset Class
class BrainTumorDataset(Dataset):
    def __init__(self, data, transform):
        super(BrainTumorDataset, self).__init__()
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx, 0], self.data[idx, 1]
        img = Image.open(img_path).convert("RGB")
        img = self.transform(np.array(img))
        return img, label

# Hyperparameters & Transforms
IMG_SIZE = 224
BATCH = 16
OUT_SIZE = len(index_label)

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Splitting Data
train, val = train_test_split(df.values, random_state=42, test_size=0.25)
test, val = train_test_split(val, random_state=42, test_size=0.5)

train_dl = DataLoader(BrainTumorDataset(train, transform), batch_size=BATCH, shuffle=True)
val_dl = DataLoader(BrainTumorDataset(val, transform), batch_size=BATCH, shuffle=False)

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Path to dataset files: /kaggle/input/brain-tumor-mri-dataset


# Teacher Model Setup and Training (ResNet50)

In [15]:
class ResNetWrapperAT(nn.Module):
    def __init__(self, model):
        super(ResNetWrapperAT, self).__init__()
        self.model = model

    def forward(self, x, return_features=False):
        # Passage manuel pour intercepter les couches intermédiaires
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)

        f1 = self.model.layer1(x)
        f2 = self.model.layer2(f1)
        f3 = self.model.layer3(f2)
        f4 = self.model.layer4(f3)

        x = self.model.avgpool(f4)
        x = torch.flatten(x, 1)
        logits = self.model.fc(x) # Logits bruts pour une meilleure stabilité en KD

        if return_features:
            return logits, [f1, f2, f3, f4]
        return logits

def at_loss(student_feats, teacher_feats):
    """Calcule la perte de transfert d'attention entre les couches."""
    loss = 0
    for s, t in zip(student_feats, teacher_feats):
        # On calcule la carte d'attention (somme des carrés sur les canaux)
        s_map = nn.functional.normalize(s.pow(2).mean(1).view(s.size(0), -1), p=2, dim=1)
        t_map = nn.functional.normalize(t.pow(2).mean(1).view(t.size(0), -1), p=2, dim=1)
        loss += (s_map - t_map).pow(2).mean()
    return loss / len(student_feats)

resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = nn.Linear(resnet.fc.in_features, OUT_SIZE)
teacher_model = ResNetWrapperAT(resnet).to(device)

# # Training configurations
# EPOCHS = 50
# LR = 0.1
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(teacher_model.parameters(), lr=LR)
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# # Training Loop
# best_model = deepcopy(teacher_model)
# best_acc = 0
# train_loss, train_acc, val_loss, val_acc = [], [], [], []

# for epoch in range(1, EPOCHS + 1):
#     teacher_model.train()
#     diff, acc, total = 0, 0, 0

#     for data, target in train_dl:
#         optimizer.zero_grad()
#         data, target = data.to(device), target.to(device)

#         out = teacher_model(data)
#         loss = criterion(out, target)

#         loss.backward()
#         optimizer.step()

#         diff += loss.item()
#         acc += (out.argmax(1) == target).sum().item()
#         total += out.size(0)

#     train_loss.append(diff / total)
#     train_acc.append(acc / total)

#     # Validation Phase
#     teacher_model.eval()
#     diff, acc, total = 0, 0, 0

#     with torch.no_grad():
#         for data, target in val_dl:
#             data, target = data.to(device), target.to(device)
#             out = teacher_model(data)
#             loss = criterion(out, target)
#             diff += loss.item()
#             acc += (out.argmax(1) == target).sum().item()
#             total += out.size(0)

#     val_loss.append(diff / total)
#     val_acc.append(acc / total)

#     # Save best model
#     if val_acc[-1] >= best_acc:
#         best_acc = val_acc[-1]
#         best_model = deepcopy(teacher_model)

#     scheduler.step()
#     print(f"Epoch {epoch} | Train Loss: {train_loss[-1]:.4f} | Train Acc: {train_acc[-1]:.4f} | Val Loss: {val_loss[-1]:.4f} | Val Acc: {val_acc[-1]:.4f}")

# plot_training_history(train_loss, val_loss, train_acc, val_acc, title="ResNet50 Teacher")

# Configuration des chemins Drive

# Initialisation du Teacher (ResNet50)
teacher_path = "/content/saved_models/teacher-resnet50.pth"
base_teacher = resnet50(weights=None)
base_teacher.fc = nn.Linear(base_teacher.fc.in_features, OUT_SIZE)
teacher_model = ResNetWrapperAT(base_teacher).to(device)

# Chargement des poids
if os.path.exists(teacher_path):
    teacher_model.load_state_dict(torch.load(teacher_path, map_location=device))
    teacher_model.eval()
    print("Poids du Teacher chargés. Prêt pour la distillation AT.")
else:
    print("Erreur : Fichier introuvable sur le Drive.")

Erreur : Fichier introuvable sur le Drive.


# Attention based functions

In [16]:
def compute_attention_map(feature_map):
    """
    Transforme un tenseur de caractéristiques (B, C, H, W)
    en une carte d'attention 2D normalisée (B, H, W).
    """
    # On met au carré et on fait la moyenne sur la dimension des canaux (dim=1)
    attention = torch.mean(feature_map.pow(2), dim=1)

    # On normalise la carte (L2-norm) pour pouvoir comparer le Teacher et le Student
    # même s'ils n'ont pas la même amplitude d'activation
    attention = nn.functional.normalize(attention, p=2, dim=(1, 2))
    return attention

def attention_transfer_loss(student_features, teacher_features):
    """
    Calcule l'erreur quadratique moyenne (MSE) entre les cartes d'attention.
    """
    loss = 0
    # On compare feat1_S avec feat1_T, etc.
    for sf, tf in zip(student_features, teacher_features):
        student_attention = compute_attention_map(sf)
        teacher_attention = compute_attention_map(tf)
        loss += nn.functional.mse_loss(student_attention, teacher_attention)

    # On retourne la perte moyenne sur les 4 blocs
    return loss / len(student_features)

# Student Model Setup and Training (Knowledge Distillation)

In [ ]:
# Initialisation de l'Étudiant (ResNet18)
base_student = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
base_student.fc = nn.Linear(base_student.fc.in_features, OUT_SIZE)
student_model = ResNetWrapperAT(base_student).to(device)

optimizer_student = torch.optim.Adam(student_model.parameters(), lr=0.001)

# Hyperparamètres
T = 3.0       # Température
alpha = 0.5   # Poids KD (Logits)
beta = 1000.0 # Poids AT (Features) - On utilise un gros poids car la MSE AT est très petite

print("Début de l'entraînement Student avec Attention Transfer...")

for epoch in range(1, 51): # 3 époques comme dans votre code
    student_model.train()
    total_loss = 0

    for data, target in train_dl:
        data, target = data.to(device), target.to(device)
        optimizer_student.zero_grad()

        # Inférence Teacher (figé)
        with torch.no_grad():
            t_logits, t_feats = teacher_model(data, return_features=True)

        # Inférence Student
        s_logits, s_feats = student_model(data, return_features=True)

        # 1. Perte de classification classique
        loss_ce = nn.functional.cross_entropy(s_logits, target)

        # 2. Perte de distillation (Soft Logits)
        loss_kd = nn.KLDivLoss(reduction='batchmean')(
            nn.functional.log_softmax(s_logits / T, dim=1),
            nn.functional.softmax(t_logits / T, dim=1)
        ) * (T * T)

        # 3. Perte d'Attention Transfer
        loss_attention = at_loss(s_feats, t_feats)

        # Somme pondérée
        loss = (1 - alpha) * loss_ce + alpha * loss_kd + beta * loss_attention

        loss.backward()
        optimizer_student.step()
        total_loss += loss.item()

    print(f"Epoch {epoch} | Loss: {total_loss/len(train_dl):.4f} | AT Contrib: {beta*loss_attention:.4f}")

# Sauvegarde finale du petit génie


Début de l'entraînement Student avec Attention Transfer...


# Model Evaluation & Inference

In [ ]:
# 6. Evaluation
def predict(img_path, model):
    img = Image.open(img_path).convert("RGB")
    img_tensor = transform(np.array(img)).view(1, 3, 224, 224).to(device)

    model.eval()
    with torch.no_grad():
        out = model(img_tensor)

    pred_idx = out.argmax(1).item()
    confidence = round(out[0][pred_idx].item() * 100, 2)
    return pred_idx, confidence

# Evaluate using the chosen model (using best_model / teacher here as per original code)
truth = []
preds = []

np.random.shuffle(test)

for i in range(test.shape[0]):
    truth.append(test[i, 1])
    pred, _ = predict(test[i, 0], best_model)
    preds.append(pred)

score = accuracy_score(truth, preds)
label_names = [index_label[i] for i in range(OUT_SIZE)]

print("\n--- Classification Report ---")
print(classification_report(truth, preds, target_names=label_names))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(truth, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.title(f"Test Set Accuracy: {score:.2%}")
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()



# Student evaluation

# 7. Student Model Evaluation
truth_student = []
preds_student = []

print("Starting Student Model evaluation on test set...")

for i in range(test.shape[0]):
    img_path, label = test[i, 0], test[i, 1]

    # Image preprocessing
    img = Image.open(img_path).convert("RGB")
    img_tensor = transform(np.array(img)).view(1, 3, 224, 224).to(device)

    # Student Inference
    student_model.eval()
    with torch.no_grad():
        # We use the wrapper which includes the softmax
        out = student_model(img_tensor)

    pred_idx = out.argmax(1).item()

    truth_student.append(label)
    preds_student.append(pred_idx)

# Calculation of metrics
score_student = accuracy_score(truth_student, preds_student)
label_names = [index_label[i] for i in range(OUT_SIZE)]

print("\n--- Student Model Classification Report (ResNet18) ---")
print(classification_report(truth_student, preds_student, target_names=label_names))

# Plotting the Student Confusion Matrix
plt.figure(figsize=(8, 6))
cm_student = confusion_matrix(truth_student, preds_student)
sns.heatmap(cm_student, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_names, yticklabels=label_names)

plt.title(f"Student Model Test Accuracy: {score_student:.2%}")
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
import time

# 8. Inference Speed & Model Complexity Comparison
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def measure_inference_speed(model, device, input_size=(1, 3, 224, 224), iterations=100):
    model.eval()
    dummy_input = torch.randn(*input_size).to(device)

    # Warm-up (to initialize CUDA kernels)
    for _ in range(10):
        _ = model(dummy_input)

    torch.cuda.synchronize() if device == "cuda" else None
    start_time = time.time()

    with torch.no_grad():
        for _ in range(iterations):
            _ = model(dummy_input)
            if device == "cuda":
                torch.cuda.synchronize()

    end_time = time.time()
    total_time = end_time - start_time
    avg_latency = (total_time / iterations) * 1000 # in milliseconds
    fps = iterations / total_time

    return avg_latency, fps

# Calculations
params_teacher = count_parameters(teacher_model)
params_student = count_parameters(student_model)

latency_t, fps_t = measure_inference_speed(teacher_model, device)
latency_s, fps_s = measure_inference_speed(student_model, device)

# Results Summary
print(f"{'Metric':<20} | {'Teacher (ResNet50)':<20} | {'Student (ResNet18)':<20}")
print("-" * 65)
print(f"{'Parameters':<20} | {params_teacher/1e6:<18.2f}M | {params_student/1e6:<18.2f}M")
print(f"{'Latency (ms/img)':<20} | {latency_t:<18.2f} | {latency_s:<18.2f}")
print(f"{'Throughput (FPS)':<20} | {fps_t:<18.2f} | {fps_s:<18.2f}")

# Visualization
labels = ['Teacher (ResNet50)', 'Student (ResNet18)']
latencies = [latency_t, latency_s]
fps_values = [fps_t, fps_s]

fig, ax1 = plt.subplots(figsize=(10, 6))

# Latency Bar Chart
color = 'tab:blue'
ax1.set_xlabel('Models')
ax1.set_ylabel('Latency (ms) - Lower is better', color=color)
ax1.bar(labels, latencies, color=color, alpha=0.6, width=0.4, label='Latency')
ax1.tick_params(axis='y', labelcolor=color)

# FPS Line Chart (Twin axis)
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('FPS - Higher is better', color=color)
ax2.plot(labels, fps_values, color=color, marker='o', linewidth=3, label='FPS')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Comparison of Inference Performance')
plt.show()